# Les 05 - Agentic RAG


## Installatie

Deze notebook demonstreert het Agentic RAG (Retrieval-Augmented Generation) patroon met behulp van het Microsoft Agent Framework.

**Vereisten:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — uw Azure AI Search service-eindpunt
- `AZURE_SEARCH_API_KEY` — uw Azure AI Search API-sleutel
- Azure OpenAI-implementatie geconfigureerd via omgevingsvariabelen
- Azure CLI geverifieerd (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Wat is Agentic RAG?

Traditionele RAG volgt een vaste pijplijn: documenten ophalen en vervolgens een antwoord genereren. **Agentic RAG** gaat verder door de agent autonomie te geven om te beslissen **wanneer** en **hoe** informatie opgehaald wordt.

Met Agentic RAG kan de agent:
- **Beslissen** of ophalen van informatie nodig is voordat een vraag wordt beantwoord
- **Kiezen** welke databron of welk hulpmiddel geraadpleegd wordt
- **Beoordelen** of de opgehaalde resultaten voldoende zijn en zo nodig vervolgopvragingen doen
- **Combineren** van informatie uit meerdere opvragingsstappen tot een samenhangend antwoord

Dit maakt de agent flexibeler en nauwkeuriger in vergelijking met een statische haal-op-en-genereer-pijplijn.


## Een zoektool maken

In Agentic RAG worden externe gegevensbronnen verpakt als **tools** die de agent op aanvraag kan oproepen. Hierdoor kan de agent retrieval behandelen als gewoon een andere actie die het kan uitvoeren, in plaats van een verplichte stap.

Hieronder definiëren we een reiskennisbank en maken deze beschikbaar als een tool die de agent kan aanroepen om bestemmingsinformatie op te zoeken.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Het bouwen van de RAG-agent

Nu maken we een agent die de instructie krijgt om **altijd informatie op te zoeken voordat hij antwoord geeft**. De agent gebruikt de tool `search_travel_knowledge` om zijn antwoorden te baseren op de kennisbank in plaats van te vertrouwen op zijn eigen trainingsgegevens.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iteratieve Ophaling — Het Maker-Checker Patroon

Een belangrijk voordeel van Agentic RAG is **iteratieve ophaling**. De agent kan meerdere zoekrondes uitvoeren om zijn initiële bevindingen te verifiëren, te verfijnen of uit te breiden — vergelijkbaar met een "maker-checker" werkwijze:

1. **Maker stap**: De agent haalt initiële informatie op en stelt een antwoord op.
2. **Checker stap**: De agent voert extra ophalingen uit om details te verifiëren of hiaten op te vullen.

Hieronder wordt de agent een vraag gesteld die het vergelijken van meerdere bestemmingen vereist, wat de agent aanspoort om meerdere keren te zoeken.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Samenvatting

In deze les heb je geleerd hoe je een **Agentic RAG**-systeem bouwt met het Microsoft Agent Framework:

- **Agentic RAG** laat agenten autonoom beslissen wanneer ze informatie ophalen, waardoor ophalen dynamisch wordt in plaats van vast.
- **Tools als databronnen**: Externe kennisbanken (zoals Azure AI Search) worden verpakt als tools die de agent kan aanroepen.
- **Iteratief ophalen**: Het maker-checker patroon stelt de agent in staat om meerdere ophaalrondes uit te voeren — zoeken, verifiëren en verfijnen — voordat een definitief antwoord wordt gegeven.

In productie zou je de in-memory `TRAVEL_KNOWLEDGE_BASE` vervangen door een echte Azure AI Search-index om grootschalige reisdocumentopvraging aan te kunnen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat automatische vertalingen fouten of onjuistheden kunnen bevatten. Het oorspronkelijke document in de originele taal wordt beschouwd als de gezaghebbende bron. Voor cruciale informatie wordt een professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
